# Africa-USA Maritime Port Trade & Shipping Connectivity — Analysis

This notebook explores maritime shipping connectivity, container port throughput,
logistics performance and trade flows for **17 African coastal nations** and the
**USA**, using World Bank Open Data.

**Data store:** `data/maritime_ports.db` (SQLite), produced by the ETL pipeline
(`ingest → transform → load`). Run the pipeline first if the database is missing.

**Sections**
1. Setup & data loading
2. Exploratory data analysis (EDA)
3. Visualizations (6)
4. Key findings


In [ ]:
import os
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.max_columns", None)


## 1. Data loading

Connect to the SQLite database and load the main table.

In [ ]:
# Resolve the database path relative to the notebook location.
DB_CANDIDATES = [
    os.path.join("..", "data", "maritime_ports.db"),
    os.path.join("data", "maritime_ports.db"),
    "maritime_ports.db",
]
DB_PATH = next((p for p in DB_CANDIDATES if os.path.exists(p)), DB_CANDIDATES[0])
print("Using database:", os.path.abspath(DB_PATH))

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("SELECT * FROM maritime_indicators", conn)
meta = pd.read_sql_query("SELECT * FROM country_metadata", conn)
yearly = pd.read_sql_query("SELECT * FROM yearly_summary", conn)
conn.close()

print("maritime_indicators:", df.shape)
df.head()


## 2. Exploratory data analysis

Basic shape, coverage and summary statistics.

In [ ]:
print("Countries:", df["country"].nunique())
print("Year range:", int(df["year"].min()), "-", int(df["year"].max()))
print("\nNon-null counts per indicator:")
print(df[[
    "liner_shipping_index", "container_throughput_teu",
    "merchandise_trade_pct_gdp", "lpi_score", "exports_usd",
]].notna().sum())


In [ ]:
df.describe(include=[np.number]).T

## 3. Visualizations

### 3.1 Top 10 African countries by average liner shipping connectivity index

In [ ]:
africa = df[df["region"] == "Africa"]
conn_rank = (
    africa.dropna(subset=["liner_shipping_index"])
    .groupby("country")["liner_shipping_index"].mean()
    .sort_values(ascending=False).head(10)
)

plt.figure()
ax = sns.barplot(x=conn_rank.values, y=conn_rank.index, palette="viridis")
ax.set_title("Top 10 African Countries by Avg Liner Shipping Connectivity Index")
ax.set_xlabel("Average Liner Shipping Connectivity Index")
ax.set_ylabel("Country")
plt.tight_layout()
plt.show()
conn_rank


### 3.2 Container port throughput trends for top 5 countries

In [ ]:
focus = ["South Africa", "Egypt, Arab Rep.", "Morocco", "Nigeria", "Kenya"]
# Fall back to name-contains matching to be robust to World Bank naming.
def match_country(name):
    for f in focus:
        key = f.split(",")[0]
        if key.lower() in str(name).lower():
            return True
    return False

teu = df[df["country"].apply(match_country)].dropna(subset=["container_throughput_teu"])
teu = teu[(teu["year"] >= 2015) & (teu["year"] <= 2024)]

plt.figure()
for country, grp in teu.groupby("country"):
    grp = grp.sort_values("year")
    plt.plot(grp["year"], grp["container_throughput_teu"] / 1e6,
             marker="o", label=country)
plt.title("Container Port Throughput Trends 2015-2024 (Top 5 Countries)")
plt.xlabel("Year")
plt.ylabel("Container Throughput (million TEU)")
plt.legend()
plt.tight_layout()
plt.show()


### 3.3 LPI score vs liner shipping connectivity (scatter)

In [ ]:
# Latest non-null LPI and connectivity per country.
def latest(frame, col):
    sub = frame.dropna(subset=[col]).sort_values("year")
    return sub.groupby(["country", "iso3"]).tail(1)[["country", "iso3", col]]

lpi_latest = latest(df, "lpi_score")
conn_latest = latest(df, "liner_shipping_index")
scatter = lpi_latest.merge(conn_latest, on=["country", "iso3"], how="inner")

plt.figure()
ax = sns.scatterplot(
    data=scatter, x="lpi_score", y="liner_shipping_index",
    hue="country", s=120, legend=False,
)
for _, r in scatter.iterrows():
    ax.annotate(r["iso3"], (r["lpi_score"], r["liner_shipping_index"]),
                fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.set_title("LPI Score vs Liner Shipping Connectivity (latest per country)")
ax.set_xlabel("Logistics Performance Index (score)")
ax.set_ylabel("Liner Shipping Connectivity Index")
plt.tight_layout()
plt.show()


### 3.4 Heatmap: merchandise trade (% of GDP) by country and year

In [ ]:
pivot = (
    df.dropna(subset=["merchandise_trade_pct_gdp"])
    .pivot_table(index="country", columns="year",
                 values="merchandise_trade_pct_gdp", aggfunc="mean")
)
# Keep recent years for readability.
recent_years = sorted([c for c in pivot.columns if c >= 2015])
pivot = pivot[recent_years]

plt.figure(figsize=(12, 8))
sns.heatmap(pivot, cmap="YlGnBu", annot=False, linewidths=0.3,
            cbar_kws={"label": "Merchandise trade (% of GDP)"})
plt.title("Merchandise Trade (% of GDP) by Country and Year")
plt.xlabel("Year")
plt.ylabel("Country")
plt.tight_layout()
plt.show()


### 3.5 Africa vs USA exports comparison (latest year)

In [ ]:
exp = df.dropna(subset=["exports_usd"]).copy()
latest_exp = exp.sort_values("year").groupby(["country", "iso3", "region"]).tail(1)
latest_exp = latest_exp.sort_values("exports_usd", ascending=False)

plt.figure()
colors = latest_exp["region"].map({"USA": "crimson", "Africa": "steelblue"})
plt.barh(latest_exp["country"], latest_exp["exports_usd"] / 1e9, color=colors)
plt.title("Exports of Goods & Services (latest year) — Africa vs USA")
plt.xlabel("Exports (US$ billion)")
plt.ylabel("Country")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()
latest_exp[["country", "region", "year", "exports_usd"]].head(10)


### 3.6 Container throughput growth rate by country (Plotly bar)

In [ ]:
grow = df.dropna(subset=["container_throughput_teu"]).sort_values("year")
first = grow.groupby(["country", "iso3"]).first().rename(
    columns={"container_throughput_teu": "first_teu", "year": "first_year"})
last = grow.groupby(["country", "iso3"]).last().rename(
    columns={"container_throughput_teu": "last_teu", "year": "last_year"})
growth = first.join(last, lsuffix="_f", rsuffix="_l").reset_index()
growth = growth[growth["first_teu"] > 0]
growth["growth_pct"] = 100 * (growth["last_teu"] - growth["first_teu"]) / growth["first_teu"]
growth = growth.sort_values("growth_pct", ascending=False)

fig = px.bar(
    growth, x="growth_pct", y="country", orientation="h",
    color="growth_pct", color_continuous_scale="Tealrose",
    title="Container Port Throughput Growth Rate by Country (first vs latest year)",
    labels={"growth_pct": "Total growth (%)", "country": "Country"},
)
fig.update_layout(height=650, yaxis={"categoryorder": "total ascending"})
fig.show()


## 4. Key findings

- **South Africa, Egypt and Morocco** consistently record the highest liner
  shipping connectivity in Africa, reflecting their positions on major global
  trade lanes.
- **Nigeria** dominates West African container throughput thanks to its large
  domestic economy.
- The **USA** posts the highest Logistics Performance Index of all countries
  analyzed — a clear benchmark gap versus most African ports.
- **Merchandise trade as a share of GDP** is markedly higher for many African
  economies than for the USA, even though the USA's absolute export volumes dwarf
  those of every African nation.
- The scatter of **LPI vs shipping connectivity** shows a positive association:
  better logistics performance tends to accompany stronger liner connectivity.

These patterns highlight both the strengths of Africa's leading maritime hubs
and the connectivity/logistics gap that remains relative to the USA benchmark.
